# Cogni Chunk 📚: Multi-Agent RAG System
## Intelligent Technical Document Analyst

This notebook presents a self-contained **Multi-Agent Retrieval-Augmented Generation (RAG)** pipeline powered by **LangChain** and **LangGraph**.

Instead of using a single monolithic class, we orchestrate a StateGraph workflow with two specialized agents:
1. **🔍 Researcher Agent**: Responsible for searching, scoring, and retrieving the most relevant chunks of data from the document.
2. **✍️ Writer Agent**: Responsible for synthesizing a grounded, easy-to-read answer based solely on the evidence provided by the Researcher.

All code is self-contained within this notebook, with no external RAG engine imports.

In [20]:
# Ensure required dependencies are installed
# !pip install -q langchain langchain-community langgraph nltk pypdf langchain-text-splitters

import re
import math
from pathlib import Path
from typing import List, Iterable, Dict, Set, TypedDict

import nltk
from nltk.corpus import stopwords
from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langgraph.graph import StateGraph, START, END

# Fetch stopwords dynamically from NLTK
nltk.download('stopwords', quiet=True)
STOPWORDS = set(stopwords.words('english'))

def tokenize(text: str) -> List[str]:
    return [token for token in re.findall(r"[a-zA-Z0-9\-]+", text.lower()) if token not in STOPWORDS]

def build_ngrams(tokens: List[str], size: int = 2) -> set:
    return {" ".join(tokens[index : index + size]) for index in range(len(tokens) - size + 1)}


In [21]:
def split_markdown_sections(markdown_text: str) -> List[Document]:
    """Splits a markdown document into LangChain Documents based on headings."""
    sections, stack, lines = [], [], []

    def flush_section() -> None:
        content = "\n".join(lines).strip()
        if content and len(tokenize(content)) > 0:
            title = stack[-1] if stack else "Document Overview"
            sections.append(Document(page_content=content, metadata={"title": title, "heading_path": " > ".join(stack) or title}))
        lines.clear()

    for raw_line in markdown_text.splitlines():
        if match := re.match(r"^(#{1,6})\s+(.*)$", raw_line.rstrip()):
            flush_section()
            stack = stack[:len(match.group(1)) - 1] + [match.group(2).strip()]
        else:
            lines.append(raw_line.rstrip())

    flush_section()
    return sections


In [22]:
# Load Markdown Document
DOC_PATH = Path('technical_doc.md')
document_text = DOC_PATH.read_text(encoding='utf-8')
md_docs = split_markdown_sections(document_text)

# Load and chunk PDF Document
pdf_path = Path('geometry_notes.pdf')
pdf_loader = PyPDFLoader(str(pdf_path))
pdf_pages = pdf_loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
pdf_docs = text_splitter.split_documents(pdf_pages)

# Standardize PDF Metadata
for i, doc in enumerate(pdf_docs, 1):
    doc.metadata["title"] = f"Geometry Notes - Segment {i}"
    doc.metadata["heading_path"] = f"Geometry Notes > Segment {i}"

all_docs = md_docs + pdf_docs
print(f"Markdown sections extracted: {len(md_docs)}")
print(f"PDF segments extracted: {len(pdf_docs)}")
print(f"Total documents loaded: {len(all_docs)}")


Markdown sections extracted: 24
PDF segments extracted: 40
Total documents loaded: 64


## LangGraph Multi-Agent Architecture

Instead of a monolith, we build a stateful graph where nodes are acting as distinct agents:

### 1. Researcher Node
Computes TF-IDF scores and matches query terms against LangChain Document chunks. Its job is purely analytical: find the best evidence and append it to the graph state.

### 2. Writer Node
Takes the state loaded by the Researcher, selects the most relevant sentences, synthesizes an answer, and yields the final payload.

In [23]:
class ResearcherAgent:
    """Analytical module used by the researcher node."""
    def __init__(self, docs: List[Document]):
        self.docs = docs
        self._idf = self._compute_idf(self.docs)

    @staticmethod
    def _compute_idf(docs: Iterable[Document]) -> Dict[str, float]:
        from collections import Counter
        freqs = Counter(token for doc in docs for token in set(tokenize(f"{doc.metadata.get('heading_path', '')} {doc.page_content}")))
        return {tok: math.log((1 + len(list(docs))) / (1 + freq)) + 1.0 for tok, freq in freqs.items()}
    
    def is_reference_section(self, doc: Document) -> bool:
        return any(m in doc.metadata.get('heading_path', '').lower() for m in ["sample questions", "project presentation notes", "closing notes"])

    def search(self, query: str, top_k: int = 4) -> List[dict]:
        q_tokens, q_bigrams = tokenize(query), build_ngrams(tokenize(query), size=2)
        results: List[dict] = []
        
        for doc in self.docs:
            heading = doc.metadata.get('heading_path', '')
            haystack = f"{heading}\n{doc.page_content}".lower()
            chunk_tokens = tokenize(haystack)
            if not chunk_tokens or not (matched := sorted(set(q_tokens) & set(chunk_tokens))):
                continue

            c_len = len(chunk_tokens)
            score = sum((chunk_tokens.count(t) / c_len) * self._idf.get(t, 1.0) + (0.35 if t in heading.lower() else 0) + (0.12 if self._idf.get(t, 1.0) > 2.6 else 0) for t in matched)
            
            score += 0.08 if any(p in haystack for p in ["because", "falls back to", "results in", "latency", "fails", "incident", "therefore", "theorem", "proof"]) else 0.0
            score += (len(matched) / len(set(q_tokens))) * 0.18 + len(q_bigrams & build_ngrams(chunk_tokens, size=2)) * 0.22 + min(c_len, 160) / 4000
            score *= 0.35 if self.is_reference_section(doc) else 1.0
                
            results.append({"doc": doc, "score": score, "matched_terms": matched})

        return sorted(results, key=lambda item: item["score"], reverse=True)[:top_k]

researcher = ResearcherAgent(all_docs)


In [24]:
class AgentState(TypedDict):
    query: str
    retrieved_docs: List[dict]
    answer: str
    confidence: str

def research_node(state: AgentState):
    print(f"\n[Researcher Node] Searching for evidence for: '{state['query']}'...")
    return {"retrieved_docs": researcher.search(state["query"], top_k=4)}

def writer_node(state: AgentState):
    results = state.get("retrieved_docs", [])
    print(f"[Writer Node] Synthesizing answer based on {len(results)} retrieved chunks...")
    
    if not results:
        return {"answer": "I could not find grounded evidence for that query in the current documents.", "confidence": "LOW"}

    top_score = results[0]["score"]
    valid = [r for r in results if r["score"] >= top_score * 0.45 and not researcher.is_reference_section(r["doc"])] or [results[0]]

    evidence_lines: List[str] = []
    for r in valid[:3]:
        sentences = re.split(r"(?<=[.!?])\s+", r["doc"].page_content.replace("\n", " "))
        best = [s.strip() for s in sentences if any(t in s.lower() for t in r["matched_terms"])]
        evidence_lines.extend(best[:2] if best else [r["doc"].page_content.strip()][:2])

    titles = [r["doc"].metadata.get("title", "Section") for r in valid[:3]]
    answer = f"The strongest evidence points to '{titles[0]}'. {' '.join(evidence_lines[:4]).strip()} This answer is grounded in sections about {', '.join(titles)}."
    return {"answer": answer, "confidence": "HIGH" if top_score > 0.45 else "MEDIUM"}


In [25]:
print("Building LangGraph workflow...")
workflow = StateGraph(AgentState)

workflow.add_node("researcher", research_node)
workflow.add_node("writer", writer_node)

workflow.add_edge(START, "researcher")
workflow.add_edge("researcher", "writer")
workflow.add_edge("writer", END)

# Compile the LangGraph application
rag_system = workflow.compile()
print("LangGraph RAG System compiled successfully!")


Building LangGraph workflow...
LangGraph RAG System compiled successfully!


In [26]:
demo_queries = [
    'What happens when the vector index is unavailable?',
    'Why were write-heavy dashboards slower during the replica failover test?',
    'What is the relationship between an inscribed angle and its intercepted arc?',
    'How does the altitude to the hypotenuse behave in a right triangle?',
]

for q in demo_queries:
    result = rag_system.invoke({"query": q})
    
    print("="*100)
    print(f"Query: {result['query']}")
    print(f"Confidence: {result['confidence']}")
    print(f"\nAnswer:\n{result['answer']}\n")
    print("Top Evidence Used by Researcher:")
    for idx, res in enumerate(result['retrieved_docs'], start=1):
        doc = res['doc']
        preview = re.sub(r"\s+", " ", doc.page_content.replace("\n", " ")).strip()
        heading = doc.metadata.get('heading_path', '')
        print(f"  {idx}. {heading} | Score: {res['score']:.3f} | Matched: {', '.join(res['matched_terms'])}")
        print(f"     {preview[:240]}...")
    print("="*100)



[Researcher Node] Searching for evidence for: 'What happens when the vector index is unavailable?'...
[Writer Node] Synthesizing answer based on 4 retrieved chunks...
Query: What happens when the vector index is unavailable?
Confidence: HIGH

Answer:
The strongest evidence points to '12.1 Vector Index Degradation'. If the vector index becomes unavailable, Atlas falls back to lexical retrieval only. This answer is grounded in sections about 12.1 Vector Index Degradation.

Top Evidence Used by Researcher:
  1. Technical Design Dossier: Atlas Knowledge Fabric > 12. Failure Modes and Recovery > 12.1 Vector Index Degradation | Score: 1.994 | Matched: index, unavailable, vector
     If the vector index becomes unavailable, Atlas falls back to lexical retrieval only. Recall drops on conceptual questions, but identifier-based lookups still work. The UI surfaces a reduced-confidence banner in this mode....
  2. Technical Design Dossier: Atlas Knowledge Fabric > 3. Reference Deployment Topology